In [1]:
import pandas as pd
import numpy as np
import os

# --- 경로 설정 ---
VENDOR_ANALYSIS_PATH = "../output/problem2_vendor/problem2_vendor_analysis_base.csv"
OUT_DIR = "../output/model"
MODEL_FILE_NAME = "basic_rule_model.pkl"
MODEL_PATH = os.path.join(OUT_DIR, MODEL_FILE_NAME)


# --- 1. 데이터 로드 및 모델 적용 ---
try:
    # 1. 과제 2 최종 분석 데이터 로드
    df_analysis = pd.read_csv(
        VENDOR_ANALYSIS_PATH, 
        usecols=['symbol', 'date', 'surprise_z', 'return_post_1d'],
        dtype={'surprise_z': np.float32, 'return_post_1d': np.float32}
    ).dropna(subset=['surprise_z', 'return_post_1d'])

    # 2. 모델 규칙 로드 (get_investment_decision_basic 함수를 대체)
    Z_SCORE_THRESHOLD = 2.0
    
    # 3. 모델 의사 결정 적용 (BUY/SELL/HOLD 컬럼 생성)
    df_analysis['decision'] = np.select(
        [
            df_analysis['surprise_z'] > Z_SCORE_THRESHOLD,
            df_analysis['surprise_z'] < -Z_SCORE_THRESHOLD,
        ],
        ['BUY', 'SELL'],
        default='HOLD'
    )
    
    df_signals = df_analysis[df_analysis['decision'].isin(['BUY', 'SELL'])].copy()
    
    if df_signals.empty:
        print("분석: BUY/SELL 시그널이 발생하지 않아 성능 평가를 할 수 없습니다.")
        exit()

except FileNotFoundError as e:
    print(f"❌ 오류: 필요한 파일을 찾을 수 없습니다. 경로를 확인하세요: {e}")
    exit()
except Exception as e:
    print(f"❌ 오류: 데이터 처리 중 예상치 못한 오류 발생: {e}")
    exit()


# --- 2. 정밀도 (Precision) 및 수익률 계산 ---

# 'return_post_1d'의 실제 방향을 기준으로 예측 성공 여부 판단
# BUY 성공: return_post_1d > 0
# SELL 성공: return_post_1d < 0 (주가 하락 = Short 수익)
df_signals['is_correct'] = np.select(
    [
        (df_signals['decision'] == 'BUY') & (df_signals['return_post_1d'] > 0),
        (df_signals['decision'] == 'SELL') & (df_signals['return_post_1d'] < 0)
    ],
    [1, 1],
    default=0
)

# 1. 정밀도 (Precision) 계산
df_precision = df_signals.groupby('decision')['is_correct'].agg(['sum', 'count'])
df_precision['Precision (%)'] = (df_precision['sum'] / df_precision['count']) * 100
df_precision = df_precision.rename(columns={'sum': 'Correct Predictions', 'count': 'Total Signals'})


# 2. 평균 수익률 (Average Return) 계산
df_avg_return = df_signals.groupby('decision')['return_post_1d'].mean().mul(100).round(4)
df_avg_return = df_avg_return.rename('Avg_Return_Post_1D (%)')


# --- 3. 최종 결과 출력 ---

print("\n" + "="*70)
print("🎯 기본 규칙 모델 (Z-Score) 최종 성능 평가")
print("="*70)

# 정밀도 출력
print("1. 시그널 정밀도 (Precision)")
print(df_precision[['Total Signals', 'Correct Predictions', 'Precision (%)']].to_markdown())
print("\n")

# 평균 수익률 출력
print("2. 평균 수익률 (Avg Return)")
print(df_avg_return.to_markdown())
print("="*70)

print("\n🚨 해석 가이드:")
print("1. BUY Precision이 50%를 넘어야 매수 시그널이 무작위보다 우수합니다.")
print("2. SELL Precision이 50%를 넘어야 매도 시그널이 무작위보다 우수합니다.")
print("3. Avg Return이 BUY는 양수, SELL은 음수여야 예측 방향성이 일치합니다.")


🎯 기본 규칙 모델 (Z-Score) 최종 성능 평가
1. 시그널 정밀도 (Precision)
| decision   |   Total Signals |   Correct Predictions |   Precision (%) |
|:-----------|----------------:|----------------------:|----------------:|
| BUY        |             328 |                   161 |         49.0854 |
| SELL       |             315 |                   130 |         41.2698 |


2. 평균 수익률 (Avg Return)
| decision   |   Avg_Return_Post_1D (%) |
|:-----------|-------------------------:|
| BUY        |                   0.2718 |
| SELL       |                   0.3046 |

🚨 해석 가이드:
1. BUY Precision이 50%를 넘어야 매수 시그널이 무작위보다 우수합니다.
2. SELL Precision이 50%를 넘어야 매도 시그널이 무작위보다 우수합니다.
3. Avg Return이 BUY는 양수, SELL은 음수여야 예측 방향성이 일치합니다.
